# 00 Environment Check

Use this notebook before touching the rover controls.

Goal:

1. confirm we are in the right JetCar folder
2. show the exact install commands if the environment is missing
3. confirm the important Python packages import
4. confirm CUDA and Torch are visible
5. confirm camera and serial devices exist

If a check fails here, fix that first before opening `01_motor_calibration_panel.ipynb`.


## Safety

This notebook does **not** drive the rover. It only prepares or checks the software environment and connected devices.


In [1]:
import os
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
venv_python = project_root / '.venv' / 'bin' / 'python'

print('Project root:', project_root)
print('Current Python:', sys.executable)
print('Virtualenv Python exists:', venv_python.exists())
print('Core project items:', sorted(p.name for p in project_root.iterdir() if p.name in {'docs', 'jetcar', 'notebooks', 'scripts', 'requirements.txt', 'requirements-notebook.txt'}))


Project root: /home/orin/JetCar
Current Python: /home/orin/JetCar/.venv/bin/python
Virtualenv Python exists: True
Core project items: ['docs', 'jetcar', 'notebooks', 'requirements-notebook.txt', 'requirements.txt', 'scripts']


In [ ]:
from textwrap import dedent

install_commands = dedent(f"""
cd {project_root}
python3 -m venv --system-site-packages .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install -r requirements.txt
python -m pip install -r requirements-notebook.txt
python -m pip install --force-reinstall --index-url https://pypi.jetson-ai-lab.io/jp6/cu126 torch==2.8.0 torchvision==0.23.0
python -m pip install --force-reinstall "numpy<2"
python -m pip install -e .
python -m ipykernel install --user --name jetcar --display-name "Python (jetcar)"
""").strip()

print("Run these terminal commands if the environment is missing or broken:\n")
print(install_commands)


In [ ]:
import importlib

packages = ['numpy', 'cv2', 'torch', 'torchvision', 'flask', 'ipywidgets', 'serial', 'PIL']
failures = {}

for name in packages:
    try:
        module = importlib.import_module(name)
        print(f'{name}:', getattr(module, '__version__', 'ok'))
    except Exception as exc:
        failures[name] = str(exc)
        print(f'{name}: FAILED -> {exc}')

if failures:
    raise RuntimeError(f'Missing or broken packages: {failures}')


In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
print('CUDA device count:', torch.cuda.device_count())
print('CUDA version:', torch.version.cuda)
if torch.cuda.is_available():
    print('CUDA device 0:', torch.cuda.get_device_name(0))


In [ ]:
import glob
import serial.tools.list_ports

video_devices = sorted(glob.glob('/dev/video*'))
serial_ports = [port.device for port in serial.tools.list_ports.comports()]
for fallback in ('/dev/ttyTHS1', '/dev/ttyTHS2'):
    if os.path.exists(fallback) and fallback not in serial_ports:
        serial_ports.append(fallback)

print('Video devices:', video_devices or ['none found'])
print('Serial devices:', serial_ports or ['none found'])
print('Calibration file exists:', (project_root / '.jetcar_motor_calibration.json').exists())
print('Contour settings file exists:', (project_root / '.contour_follow_settings.json').exists())


## Next Step

If the checks above look good:

- open `01_motor_calibration_panel.ipynb` to fine-tune `forward`, `rotate_left`, `rotate_right`, and `forward_nudge`
- then open `02_article_contour_follow.ipynb` to launch the new contour-based line-follow panel

Both panels reuse the same project folder and save files under the repo root.
